In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.forecasting_models import (
    prophet_forecast,
    arima_forecast,
    sarima_forecast,
    calculate_metrics
)

In [2]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final_dataset.csv"
)

df = pd.read_csv(
    DATA_PATH
)

print(df.shape)

df.head()

(515212, 31)


,date,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,sku_name,...,city_y,loyalty_segment,preferred_channel,registration_date,store_id_inventory,stock_on_hand,reorder_point,safety_stock,last_restock_date,snapshot_date
0,2021-01-01,26,1124,2961.0,1,20.08,20.08,Store,15.0,Dairy_Cheese_1124,...,Dubai,Silver,Website,2024-12-19,10,185,70,35,2025-09-27,2025-10-31
1,2021-01-01,38,1088,2507.0,1,17.99,17.99,Website,15.0,Household_Cleaning Supplies_1088,...,Dubai,Gold,Mobileapp,2023-12-17,9,241,96,48,2025-09-21,2025-10-31
2,2021-01-01,2,1093,1252.0,1,7.98,7.98,Store,0.0,Snacks_Chips_1093,...,Dubai,Gold,Mobileapp,2024-11-20,9,289,125,62,2025-09-17,2025-10-31
3,2021-01-01,25,1067,1286.0,1,42.13,42.13,Mobileapp,15.0,Grocery_Cereals_1067,...,Sharjah,Silver,Store,2024-04-02,11,303,116,58,2025-09-27,2025-10-31
4,2021-01-01,2,1043,2507.0,3,19.93,59.79,Mobileapp,15.0,Electronics_Batteries_1043,...,Dubai,Gold,Mobileapp,2023-12-17,9,137,47,23,2025-08-14,2025-10-31


In [3]:
required_columns = [
    "date",
    "quantity"
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("Required columns available.")

Required columns available.


In [4]:
df["date"] = pd.to_datetime(
    df["date"]
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "date",
        "quantity"
    ]
)

df = df.sort_values(
    "date"
)

daily_demand = (
    df.groupby("date")["quantity"]
    .sum()
    .reset_index()
)

daily_demand.head()

,date,quantity
0,2021-01-01,503
1,2021-01-02,1579
2,2021-01-03,1579
3,2021-01-04,1171
4,2021-01-05,1138


In [5]:
prophet_model, prophet_result = prophet_forecast(
    df,
    date_col="date",
    target_col="quantity",
    periods=30
)

prophet_result.tail()

19:47:22 - cmdstanpy - INFO - Chain [1] start processing
19:47:23 - cmdstanpy - INFO - Chain [1] done processing


,date,yhat,yhat_lower,yhat_upper
1790,2025-11-26,420.137660,122.072021,703.288346
1791,2025-11-27,455.424281,174.046296,753.180962
1792,2025-11-28,471.702013,178.772697,757.230478
1793,2025-11-29,684.263174,417.272967,971.143732
1794,2025-11-30,681.299418,390.807904,985.669870


In [6]:
arima_model, arima_result = arima_forecast(
    df,
    date_col="date",
    target_col="quantity",
    order=(7, 1, 2),
    periods=30
)

arima_result.head()

c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,date,forecast
0,2025-11-01,627.621710
1,2025-11-02,600.805283
2,2025-11-03,556.016462
3,2025-11-04,467.216341
4,2025-11-05,436.741748


In [7]:
sarima_model, sarima_result = sarima_forecast(
    df,
    date_col="date",
    target_col="quantity",
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    periods=30
)

sarima_result.head()

c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\mayank\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


,date,forecast
0,2025-11-01,710.013380
1,2025-11-02,685.736550
2,2025-11-03,470.965196
3,2025-11-04,466.404491
4,2025-11-05,468.276645


In [8]:
comparison = pd.DataFrame({

    "date":
        arima_result["date"],

    "ARIMA":
        arima_result["forecast"],

    "SARIMA":
        sarima_result["forecast"],

    "Prophet":
        prophet_result.tail(30)["yhat"].values

})

comparison.head()

,date,ARIMA,SARIMA,Prophet
0,2025-11-01,627.621710,710.013380,585.964460
1,2025-11-02,600.805283,685.736550,566.695185
2,2025-11-03,556.016462,470.965196,348.318223
3,2025-11-04,467.216341,466.404491,339.359777
4,2025-11-05,436.741748,468.276645,331.403450


In [9]:
comparison["Ensemble"] = (
    comparison["ARIMA"]
    + comparison["SARIMA"]
    + comparison["Prophet"]
) / 3

comparison[
    "Ensemble"
] = comparison[
    "Ensemble"
].clip(lower=0)

comparison.tail()

,date,ARIMA,SARIMA,Prophet,Ensemble
25,2025-11-26,440.766767,458.355314,420.137660,439.753247
26,2025-11-27,486.986739,479.143498,455.424281,473.851506
27,2025-11-28,571.300104,483.554984,471.702013,508.852367
28,2025-11-29,630.194473,682.064967,684.263174,665.507538
29,2025-11-30,619.389837,668.585578,681.299418,656.424944


In [10]:
output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "multi_model_forecast.csv"
)

comparison.to_csv(
    output_path,
    index=False
)

print(
    f"Saved: {output_path}"
)

Saved: C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\data\processed\multi_model_forecast.csv
